## 0. Objectifs de l'exploration

Avant de construire le pipeline ETL, on explore l'API `FlightRadarAPI` pour :

1. **Comprendre les sources** disponibles (`get_flights`, `get_airports`, `get_airlines`, `get_zones`, `get_flight_details`).
2. **Identifier le schéma** de chaque source et les champs utiles aux indicateurs demandés.
3. **Évaluer la qualité des données** (valeurs manquantes, vols au sol, codes absents) → ce qui guidera la phase de *data cleaning*.
4. **Cartographier** chaque indicateur du kata vers les champs source nécessaires.

> Note : l'API renvoie des données *temps réel* — les volumes et valeurs changent à chaque exécution.

In [ ]:
%pip install FlightRadarAPI

In [14]:
import warnings
import logging

import pandas as pd

from FlightRadarAPI import FlightRadar24API
from FlightRadarAPI.core import Countries

# L'endpoint FR24 renvoie déjà du JSON décompressé : la librairie émet un warning
# "failed to decode Content-Encoding=gzip" mais retombe correctement sur les bytes
# bruts. On réduit le bruit du logger pour garder une sortie lisible.
logging.getLogger("FlightRadarAPI").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

fr_api = FlightRadar24API()
print("API prête —", len(fr_api.get_zones()), "zones disponibles")

API prête — 9 zones disponibles


## 1. Vols en temps réel — `get_flights()`

C'est la **table de faits** centrale du pipeline. Sans `bounds`, l'API plafonne à ~1500 vols par appel. Chaque `Flight` expose une position, une vitesse, un appareil, et les codes IATA origine/destination.

In [15]:
flights = fr_api.get_flights()  # Returns a list of Flight objects
print(f"Total flights: {len(flights)}")
if flights:
    print(flights[0])

Total flights: 1500
<(GRND)  - Altitude: 791 - Ground Speed: 1 - Heading: 179>


In [16]:
# Schéma d'un objet Flight : on inspecte les attributs publics disponibles
sample = flights[0]
{k: v for k, v in vars(sample).items() if not k.startswith("_")}

{'latitude': 45.808,
 'longitude': 8.7722,
 'id': '3f6a31cd',
 'icao_24bit': 'DF0862',
 'heading': 179,
 'altitude': 791,
 'ground_speed': 1,
 'squawk': '',
 'aircraft_code': 'GRND',
 'registration': '',
 'time': 1781806886,
 'origin_airport_iata': '',
 'destination_airport_iata': '',
 'number': '',
 'airline_iata': 'N/A',
 'on_ground': 0,
 'vertical_speed': 0,
 'callsign': 'LILC',
 'airline_icao': ''}

In [17]:
# Aplatissement des objets Flight en DataFrame pandas pour l'exploration.
# (Dans le pipeline, ce sera un DataFrame Spark — ici pandas suffit pour explorer.)
flights_df = pd.DataFrame([
    {
        "id": f.id,
        "callsign": f.callsign,
        "number": f.number,
        "airline_icao": f.airline_icao,
        "airline_iata": f.airline_iata,
        "aircraft_code": f.aircraft_code,
        "registration": f.registration,
        "origin_iata": f.origin_airport_iata,
        "destination_iata": f.destination_airport_iata,
        "latitude": f.latitude,
        "longitude": f.longitude,
        "altitude": f.altitude,
        "ground_speed": f.ground_speed,
        "heading": f.heading,
        "on_ground": f.on_ground,
        "vertical_speed": f.vertical_speed,
    }
    for f in flights
])
print(flights_df.shape)
flights_df.head()

(1500, 16)


,id,callsign,number,airline_icao,airline_iata,aircraft_code,registration,origin_iata,destination_iata,latitude,longitude,altitude,ground_speed,heading,on_ground,vertical_speed
0,3f6a31cd,LILC,,,N/A,GRND,,,,45.8080,8.7722,791,1,179,0,0
1,4030f95e,HBAL790,,,N/A,BALL,N870TH,,,39.7755,-84.4758,57500,5,68,0,0
2,40361d83,DKMFI,,,N/A,GLID,D-KMFI,,,48.6162,9.4733,1158,1,89,0,0
3,4036f99c,N807XR,,RPN,N/A,SHIP,N807XR,,,24.7020,-81.5064,4000,1,90,0,0
4,4038ee1e,STM32,,,N/A,GLID,STM32,,,-33.3779,-70.5788,2382,2,109,0,0


In [18]:
# --- Qualité des données : ce que la phase de data cleaning devra traiter ---
empty = flights_df.replace("", pd.NA)

# 1) Taux de valeurs manquantes / vides par colonne
missing = empty.isna().mean().mul(100).round(1).sort_values(ascending=False)
print("Taux de valeurs manquantes (%):")
print(missing.to_string())

# 2) Vols au sol (on_ground) — souvent à exclure des indicateurs "vols en cours"
on_ground = flights_df["on_ground"].fillna(0).astype(int)
print(f"\nVols au sol : {on_ground.sum()} / {len(flights_df)} ({on_ground.mean()*100:.1f}%)")

# 3) Vols sans origine OU destination — inexploitables pour les trajets
no_route = (empty["origin_iata"].isna() | empty["destination_iata"].isna()).mean()
print(f"Vols sans origine/destination complète : {no_route*100:.1f}%")

# 4) Codes compagnie / appareil manquants
print(f"Sans airline_icao  : {empty['airline_icao'].isna().mean()*100:.1f}%")
print(f"Sans aircraft_code : {empty['aircraft_code'].isna().mean()*100:.1f}%")

Taux de valeurs manquantes (%):
number              14.1
destination_iata    13.7
airline_icao        12.4
origin_iata          9.0
registration         4.1
callsign             0.3
aircraft_code        0.1
id                   0.0
airline_iata         0.0
latitude             0.0
longitude            0.0
altitude             0.0
ground_speed         0.0
heading              0.0
on_ground            0.0
vertical_speed       0.0

Vols au sol : 149 / 1500 (9.9%)
Vols sans origine/destination complète : 13.7%
Sans airline_icao  : 12.4%
Sans aircraft_code : 0.1%


## 2. Aéroports — `get_airports(countries)`

`get_airports` attend une **liste de membres de l'enum `Countries`** (pas des chaînes). Renvoie nom, IATA/ICAO, pays et position de chaque aéroport — une **table de dimension** pour rattacher un vol à un pays / continent.

In [19]:
Countries

<enum 'Countries'>

In [20]:
# get_airports attend des membres de l'enum Countries, pas des chaînes de caractères.
airports = fr_api.get_airports(countries=[Countries.FRANCE, Countries.ITALY])
print(f"Total aéroports (FR + IT) : {len(airports)}")
airports[0]

Total aéroports (FR + IT) : 241


<(LFBA) Agen La Garenne Airport - Altitude: None - Latitude: 44.174709 - Longitude: 0.590619>

In [21]:
airports_df = pd.DataFrame([
    {
        "name": a.name,
        "iata": a.iata,
        "icao": a.icao,
        "country": a.country,
        "latitude": a.latitude,
        "longitude": a.longitude,
        "altitude": a.altitude,
    }
    for a in airports
])
print(airports_df.shape)
print("\nAéroports par pays :")
print(airports_df["country"].value_counts())
airports_df.head()

(241, 7)

Aéroports par pays :
country
France    162
Italy      79
Name: count, dtype: int64


,name,iata,icao,country,latitude,longitude,altitude
0,Agen La Garenne Airport,AGF,LFBA,France,44.174709,0.590619,None
1,Aix-en-Provence Airport,QXB,LFMA,France,43.505264,5.367415,None
2,Ajaccio Napoleon Bonaparte Airport,AJA,LFKJ,France,41.923882,8.802500,None
3,Albert Amiens Henry Potez International Airport,BYF,LFAQ,France,49.970394,2.698475,None
4,Albertville Airport,QFA,LFKA,France,45.626598,6.329078,None


## 3. Compagnies aériennes — `get_airlines()`

Référentiel des compagnies : `Name`, `ICAO`, `IATA`, `n_aircrafts`. Sert à traduire le `airline_icao` des vols en nom lisible et constitue la dimension "compagnie" des indicateurs.

In [22]:
airlines = fr_api.get_airlines()
airlines

[{'Name': '21 Air', 'ICAO': 'CSB', 'IATA': '2I', 'n_aircrafts': 3},
 {'Name': '247 Aviation', 'ICAO': 'EMC', 'IATA': None, 'n_aircrafts': 10},
 {'Name': '2Excel Aviation', 'ICAO': 'BRO', 'IATA': None, 'n_aircrafts': 24},
 {'Name': '30 West Jets', 'ICAO': 'LVA', 'IATA': None, 'n_aircrafts': 1},
 {'Name': '4Airways', 'ICAO': 'DAK', 'IATA': None, 'n_aircrafts': 2},
 {'Name': '748 Air Services', 'ICAO': 'IHO', 'IATA': None, 'n_aircrafts': 7},
 {'Name': '7Air Cargo', 'ICAO': 'TXG', 'IATA': None, 'n_aircrafts': 3},
 {'Name': '9 Air', 'ICAO': 'JYH', 'IATA': 'AQ', 'n_aircrafts': 27},
 {'Name': 'Abakan Air', 'ICAO': 'NKP', 'IATA': 'S5', 'n_aircrafts': 6},
 {'Name': 'Abu Dhabi Aviation',
  'ICAO': 'BAR',
  'IATA': None,
  'n_aircrafts': 29},
 {'Name': 'ABX Air', 'ICAO': 'ABX', 'IATA': 'GB', 'n_aircrafts': 13},
 {'Name': 'ACASS Ireland', 'ICAO': 'SON', 'IATA': None, 'n_aircrafts': 1},
 {'Name': 'Advance Aviation', 'ICAO': 'AJV', 'IATA': None, 'n_aircrafts': 3},
 {'Name': 'Advanced Air', 'ICAO': '

In [23]:
airlines_df = pd.DataFrame(airlines)
print(airlines_df.shape)
print("\nTop 10 compagnies par nombre d'appareils :")
print(airlines_df.sort_values("n_aircrafts", ascending=False).head(10).to_string(index=False))
airlines_df.head()

(2062, 4)

Top 10 compagnies par nombre d'appareils :
                     Name ICAO IATA  n_aircrafts
United States - Air Force  RCH None         3849
     United States - Navy  CNV None         2241
        American Airlines  AAL   AA         1657
          United Airlines  UAL   UA         1579
          Delta Air Lines  DAL   DL         1324
                   Airbus  AIB None          940
                  NetJets  EJA None          881
       Southwest Airlines  SWA   WN          799
  China Southern Airlines  CSN   CZ          723
                    FedEx  FDX   FX          704


,Name,ICAO,IATA,n_aircrafts
0,21 Air,CSB,2I,3
1,247 Aviation,EMC,None,10
2,2Excel Aviation,BRO,None,24
3,30 West Jets,LVA,None,1
4,4Airways,DAK,None,2


## 4. Zones géographiques — `get_zones()` & `get_bounds()`

Les zones définissent des **boîtes englobantes** (`bounds`) qu'on passe à `get_flights(bounds=...)` pour cibler une région et dépasser la limite globale de ~1500 vols. C'est la clé pour **paginer la collecte par région** dans le pipeline.

In [24]:
zones = fr_api.get_zones()
zones

{'europe': {'tl_y': 72.57,
  'tl_x': -16.96,
  'br_y': 33.57,
  'br_x': 53.05,
  'subzones': {'poland': {'tl_y': 56.86,
    'tl_x': 11.06,
    'br_y': 48.22,
    'br_x': 28.26},
   'germany': {'tl_y': 57.92, 'tl_x': 1.81, 'br_y': 45.81, 'br_x': 16.83},
   'uk': {'tl_y': 62.61,
    'tl_x': -13.07,
    'br_y': 49.71,
    'br_x': 3.46,
    'subzones': {'london': {'tl_y': 53.06,
      'tl_x': -2.87,
      'br_y': 50.07,
      'br_x': 3.26},
     'ireland': {'tl_y': 56.22, 'tl_x': -11.71, 'br_y': 50.91, 'br_x': -4.4}}},
   'spain': {'tl_y': 44.36, 'tl_x': -11.06, 'br_y': 35.76, 'br_x': 4.04},
   'france': {'tl_y': 51.07, 'tl_x': -5.18, 'br_y': 42.17, 'br_x': 8.9},
   'ceur': {'tl_y': 51.39, 'tl_x': 11.25, 'br_y': 39.72, 'br_x': 32.55},
   'scandinavia': {'tl_y': 72.12, 'tl_x': -0.73, 'br_y': 53.82, 'br_x': 40.67},
   'italy': {'tl_y': 47.67, 'tl_x': 5.26, 'br_y': 36.27, 'br_x': 20.64}}},
 'northamerica': {'tl_y': 75,
  'tl_x': -180,
  'br_y': 3,
  'br_x': -52,
  'subzones': {'na_n': {'tl_y'

In [25]:
# Zones de premier niveau
print("Zones de premier niveau :", list(zones.keys()))

# Exemple : convertir une zone en bounds passables à get_flights(bounds=...)
europe_bounds = fr_api.get_bounds(zones["europe"])
print("\nBounds Europe (y1,y2,x1,x2):", europe_bounds)

# Sous-zones par zone (utile pour planifier une collecte régionale fine)
for name, z in zones.items():
    subs = z.get("subzones", {})
    if subs:
        print(f"  {name}: {len(subs)} sous-zones -> {list(subs.keys())}")

Zones de premier niveau : ['europe', 'northamerica', 'southamerica', 'oceania', 'asia', 'africa', 'atlantic', 'maldives', 'northatlantic']

Bounds Europe (y1,y2,x1,x2): 72.57,33.57,-16.96,53.05
  europe: 8 sous-zones -> ['poland', 'germany', 'uk', 'spain', 'france', 'ceur', 'scandinavia', 'italy']
  northamerica: 3 sous-zones -> ['na_n', 'na_c', 'na_s']
  asia: 1 sous-zones -> ['japan']


## 5. Enrichissement par les détails de vol — `get_flight_details()`

`get_flights()` ne donne que les codes IATA. Pour obtenir le **modèle/constructeur d'avion**, le **nom de compagnie**, les **pays/continents** origine et destination (indispensables aux indicateurs), il faut appeler `get_flight_details()` par vol (coûteux → à paralléliser, cf. `details=True`).

In [26]:
if flights:
    flight = flights[0]
    flight_details = fr_api.get_flight_details(flight)
    flight.set_flight_details(flight_details)
    print("Flying to", flight.destination_airport_name)
else:
    print("No flights available to get details")

Flying to N/A


In [27]:
# Champs enrichis utiles aux indicateurs du kata (présents après set_flight_details)
if flights:
    interesting = [
        "aircraft_model", "aircraft_code",
        "airline_name", "airline_icao",
        "origin_airport_name", "origin_airport_country_name",
        "destination_airport_name", "destination_airport_country_name",
        "destination_airport_latitude", "destination_airport_longitude",
    ]
    for attr in interesting:
        print(f"  {attr:36} = {getattr(flight, attr, '<absent>')}")

  aircraft_model                       = Beacon
  aircraft_code                        = GRND
  airline_name                         = LILC Airport
  airline_icao                         = 
  origin_airport_name                  = N/A
  origin_airport_country_name          = N/A
  destination_airport_name             = N/A
  destination_airport_country_name     = N/A
  destination_airport_latitude         = N/A
  destination_airport_longitude        = N/A


## 6. Filtrage ciblé — `bounds` + `airline` + `aircraft_type`

`get_flights()` accepte des filtres serveur : compagnie (ICAO), type d'appareil et zone géographique. Utile pour valider un sous-ensemble ou répartir la charge de collecte.

In [28]:
airline_icao = "UAE"
aircraft_type = "B77W"

# On peut aussi définir une région personnalisée, ex : bounds = "73,-12,-156,38"
zone = fr_api.get_zones()["northamerica"]
bounds = fr_api.get_bounds(zone)

emirates_flights = fr_api.get_flights(
    aircraft_type=aircraft_type,
    airline=airline_icao,
    bounds=bounds,
)
print(f"Vols Emirates B77W au-dessus de l'Amérique du Nord : {len(emirates_flights)}")
for f in emirates_flights:
    print(f"  {f.callsign:>8}  {f.origin_airport_iata}->{f.destination_airport_iata}  {f.get_altitude()}")

Vols Emirates B77W au-dessus de l'Amérique du Nord : 3
    UAE229  DXB->SEA  36975 ft
     UAE3U  DXB->ORD  36000 ft
    UAE32E  DXB->BOS  14550 ft


## 7. Synthèse — cartographie indicateurs → champs source

| Indicateur du kata | Champs nécessaires | Source |
|---|---|---|
| Compagnie avec le + de vols en cours | `airline_icao` (filtré `on_ground == 0`) | `get_flights` |
| Par continent, compagnie la + active en régional (continent orig == dest) | `airline_icao`, pays/continent orig & dest | `get_flights` + `get_flight_details` |
| Vol en cours au trajet le plus long | lat/lon orig & dest (distance haversine) | `get_flight_details` |
| Par continent, longueur de vol moyenne | pays/continent + distance trajet | `get_flight_details` |
| Constructeur d'avions avec le + de vols actifs | `aircraft_code` / `aircraft_model` → mapping constructeur | `get_flights` + référentiel appareils |
| Par pays de compagnie, top 3 modèles d'avion | `airline_icao` → pays compagnie, `aircraft_code` | `get_flights` + `get_airlines` |
| **Bonus** : aéroport au + grand écart départs/arrivées | `origin_iata`, `destination_iata` | `get_flights` |

### Constats clés pour le pipeline (data cleaning & design)
- **Limite ~1500 vols** par appel global → collecter **zone par zone** (`get_bounds`) pour une couverture mondiale.
- **Vols au sol** (`on_ground`) et **origine/destination manquantes** → à filtrer pour les indicateurs "vols en cours" / trajets.
- **Pays & continent** ne sont disponibles que via `get_flight_details` (1 requête/vol) → paralléliser (`details=True`) et tolérer les échecs partiels (*fault-tolerant*).
- **Continent** non fourni directement → dériver depuis le pays (table de correspondance pays→continent à ajouter au cleaning).
- **Constructeur** non fourni → dériver du préfixe `aircraft_code` (B/A…) ou d'un référentiel modèles.
- Codes manquants (`airline_icao`, `aircraft_code`) → décider du traitement (exclusion vs catégorie "UNKNOWN").